# PulmoScan AI

### Classification de coupes de scanner thoracique par deep learning

**Auteur :** Mohammed Oughlane · Marrakech, Maroc  
**Période de réalisation :** 30/10/2026 – 03/11/2026  
**Dépôt :** [pulmoscan-ai]([URL_DE_MON_REPO])

Ce notebook entraîne un modèle **TensorFlow/Keras** qui classe des coupes axiales de scanner thoracique en trois catégories :

| Classe | Signification |
|---|---|
| `nCT` | Scanner normal. Les poumons sont visibles et ne montrent pas de signe de COVID-19. |
| `pCT` | Scanner positif. Les poumons présentent des anomalies typiques d'une pneumonie à COVID-19. |
| `NiCT` | Scanner non informatif. La coupe ne montre pas assez de poumons pour conclure. |

L'approche repose sur le **transfer learning** : MobileNetV3-Large, pré-entraîné sur ImageNet, est adapté au problème en remplaçant sa tête de classification puis en affinant l'ensemble du réseau sur les coupes scanner.

**Jeu de données :** [CT Scans for COVID-19 Classification](https://www.kaggle.com/datasets/azaemon/preprocessed-ct-scans-for-covid19) (19 685 coupes, licence CC BY 4.0), construit à partir des données iCTCF de Ning et al. (2020).

> **Projet pédagogique.** Ce notebook ne constitue pas un dispositif médical et ne doit pas servir à prendre une décision de santé.

## Exécution en local

Prérequis : Python 3.11, un compte Kaggle gratuit et environ 4 Go d'espace disque pour les données.

```bash
python3.11 -m venv .venv
source .venv/bin/activate   # Windows : .venv\Scripts\Activate.ps1
pip install -r requirements.txt

# Jeton API Kaggle : kaggle.com > Settings > API > Create New Token
mkdir -p ~/.kaggle && mv ~/Downloads/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

kaggle datasets download -d azaemon/preprocessed-ct-scans-for-covid19 -p ~/datasets/covid-ct --unzip
```

Copiez ensuite les dossiers `nCT`, `NiCT` et `pCT` des **coupes originales** dans un dossier `CT Scans` placé à côté de ce notebook.

Le notebook utilise TensorFlow/Keras. Sur Apple Silicon, décommentez `tensorflow-metal` dans `requirements.txt` pour activer le GPU Metal. Sur Windows ou Linux, TensorFlow utilise le CPU ou un GPU CUDA selon l'installation.

Pour un test rapide, gardez `SMOKE_TEST = True` dans la cellule « Paramètres » : le notebook n'exécute qu'une époque et utilise un sous-ensemble réduit des données. Passez à `False` pour l'entraînement complet de 25 époques.

**Important :** les résultats affichés dans ce notebook ne sont valides qu'après un entraînement complet. Le mode smoke test sert uniquement à vérifier que le pipeline fonctionne.

## Imports

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")

## Paramètres

Tous les réglages utiles sont regroupés dans cette cellule : emplacement des images, hyperparamètres et choix du matériel.

`SMOKE_TEST = True` limite l'entraînement à une époque et à un petit sous-ensemble des données. C'est pratique pour vérifier que l'installation fonctionne avant de lancer le run complet. Pour l'entraînement réel, passez à `False`.

In [ ]:
# Dossier contenant les sous-dossiers nCT, NiCT et pCT.
# La variable d'environnement PULMOSCAN_DATA_ROOT permet de pointer vers un autre emplacement.
DATA_ROOT      = Path(os.environ.get("PULMOSCAN_DATA_ROOT", "CT Scans"))
OUTPUT_DIR     = Path("outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
ASSETS_DIR     = Path("assets")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 16
IMG_SHAPE   = 224
EPOCHS      = 25
MODEL_NAME  = "pulmoscan_mobilenetv3"
FINE_TUNING = True
SEED        = 42

# Passez à False pour l'entraînement complet (25 époques).
SMOKE_TEST = True
if SMOKE_TEST:
    EPOCHS = 1

# Reproductibilité : graines fixées pour Python, NumPy et TensorFlow.
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Détection du matériel disponible.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

DEVICE = "GPU" if gpus else "CPU"
print(f"Appareils détectés : {[d.device_type for d in tf.config.list_physical_devices()]}")
print(f"Appareil utilisé : {DEVICE} | époques : {EPOCHS} | smoke test : {SMOKE_TEST}")

## Jeu de données

Les images sont rangées dans trois dossiers, un par classe : `nCT`, `NiCT` et `pCT`. Les deux cellules suivantes vérifient que les dossiers et les images attendus sont présents.

**Ordre des classes.** Le tri alphabétique des noms de dossiers, majuscules avant minuscules (le comportement de `image_dataset_from_directory`), donne `NiCT`, `nCT`, `pCT`. Cet ordre est enregistré explicitement dans `outputs/checkpoints/class_names.json` et réutilisé partout : notebook, entraînement, évaluation et application Streamlit.

In [ ]:
EXPECTED_CLASSES = {"nCT", "NiCT", "pCT"}

# Ordre alphabétique identique à celui de image_dataset_from_directory :
# 'N' (78) passe avant 'n' (110), donc NiCT, nCT, pCT.
class_names = sorted(EXPECTED_CLASSES)
for name in class_names:
    assert (DATA_ROOT / name).is_dir(), f"Dossier manquant : {DATA_ROOT / name}"

print("Classes :", class_names)

In [ ]:
EXPECTED_COUNTS = {"nCT": 9979, "NiCT": 5705, "pCT": 4001}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def count_images(folder):
    """Compte les images d'un dossier."""
    return sum(1 for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXTS)

for c in class_names:
    n = count_images(DATA_ROOT / c)
    print(f"{c:5s} {n:6,} images")
    if not SMOKE_TEST:
        assert n == EXPECTED_COUNTS[c], f"{c} : {n} images trouvées, {EXPECTED_COUNTS[c]} attendues"

## Découpage des données

Chaque classe est mélangée avec une graine fixe (42), puis découpée en environ 70 % d'entraînement, 15 % de validation et 15 % de test. Les effectifs exacts par classe sont fixés dans le code, ce qui rend le découpage identique à chaque exécution. Aucun fichier n'est copié : le découpage est matérialisé par trois listes de chemins.

En mode `SMOKE_TEST`, le notebook utilise un petit sous-ensemble (au plus 64 images par classe pour l'entraînement) afin de vérifier le pipeline rapidement.

In [ ]:
files_by_class = {
    c: sorted(p for p in (DATA_ROOT / c).iterdir() if p.suffix.lower() in IMAGE_EXTS)
    for c in class_names
}

rng = random.Random(SEED)
for c in class_names:
    rng.shuffle(files_by_class[c])

if SMOKE_TEST:
    target = {}
    for c in class_names:
        n = len(files_by_class[c])
        if n < 3:
            raise ValueError(f"{c} : au moins 3 images sont nécessaires en mode smoke test.")
        n_train = max(1, min(64, int(n * 0.6)))
        n_valid = max(1, min(16, int(n * 0.2)))
        n_test = max(1, n - n_train - n_valid)
        target[c] = {"train": n_train, "valid": n_valid, "test": n_test}
else:
    target = {
        "pCT":  {"train": 2800, "valid": 600,  "test": 601},
        "NiCT": {"train": 3993, "valid": 855,  "test": 857},
        "nCT":  {"train": 6985, "valid": 1496, "test": 1498},
    }

train_paths, train_labels = [], []
val_paths, val_labels = [], []
test_paths, test_labels = [], []

for idx, c in enumerate(class_names):
    files = files_by_class[c]
    t, v, s = target[c]["train"], target[c]["valid"], target[c]["test"]
    need = t + v + s
    if len(files) < need:
        raise ValueError(f"{c} : {len(files)} images trouvées, {need} nécessaires.")

    train_paths += [str(p) for p in files[:t]]
    train_labels += [idx] * t
    val_paths += [str(p) for p in files[t:t + v]]
    val_labels += [idx] * v
    test_paths += [str(p) for p in files[t + v:t + v + s]]
    test_labels += [idx] * s

class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
class_to_name = {
    "nCT":  "Scanner normal",
    "NiCT": "Scanner non informatif",
    "pCT":  "Scanner positif à la COVID-19",
}
idx_to_name = {i: class_to_name[idx_to_class[i]] for i in idx_to_class}

# L'ordre des classes est enregistré explicitement pour le notebook et l'application.
class_names_path = CHECKPOINT_DIR / "class_names.json"
with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print(f"Découpage : {len(train_paths):,} entraînement / {len(val_paths):,} validation / {len(test_paths):,} test")
print("class_to_idx :", class_to_idx)
print("Ordre enregistré :", class_names_path)

## Pipeline `tf.data`

Les images sont lues avec `tf.data`, redimensionnées à 256 × 256, puis recadrées à 224 × 224.

**Prétraitement important.** Le MobileNetV3 de Keras intègre sa propre couche de normalisation et attend des pixels bruts entre 0 et 255. On ne divise donc **pas** par 255 et on n'applique **pas** les moyennes/écarts-types d'ImageNet.

**Augmentation.** Les images d'entraînement subissent un recadrage aléatoire, un retournement horizontal et un zoom aléatoire, réalisés avec des couches de prétraitement Keras. Les images de validation et de test reçoivent uniquement un recadrage central, sans augmentation.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    """Lit une image, la redimensionne à 256 px et conserve les pixels bruts 0-255."""
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image = tf.cast(image, tf.float32)  # 0-255 : attendu par MobileNetV3 Keras
    image = tf.image.resize(image, (256, 256), method="bilinear")
    return image, label

augmentation = keras.Sequential([
    layers.RandomCrop(IMG_SHAPE, IMG_SHAPE),
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),
], name="augmentation")

def center_crop(image, label):
    """Recadrage central 224 x 224, sans augmentation."""
    image = tf.image.resize_with_crop_or_pad(image, IMG_SHAPE, IMG_SHAPE)
    return image, label

def make_dataset(paths, labels, training, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(center_crop, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds      = make_dataset(train_paths, train_labels, training=True,  shuffle=True)
train_eval_ds = make_dataset(train_paths, train_labels, training=False, shuffle=False)
val_ds        = make_dataset(val_paths,   val_labels,   training=False, shuffle=False)
test_ds       = make_dataset(test_paths,  test_labels,  training=False, shuffle=False)

xb, yb = next(iter(train_ds))
print("Batch :", xb.shape, yb.shape)
print(f"Plage des pixels : {float(tf.reduce_min(xb)):.1f} à {float(tf.reduce_max(xb)):.1f} (attendu : 0 à 255)")

## Le modèle

MobileNetV3-Large transforme chaque image en un vecteur de 960 caractéristiques. Ses dernières couches d'origine produisent 1 000 classes ImageNet. Elles sont remplacées par une tête adaptée :

960 caractéristiques → 1 280 neurones (ReLU + dropout 20 %) → 3 classes (softmax)

L'ensemble du réseau est affiné (`base_model.trainable = True`), pas seulement la nouvelle tête.

In [ ]:
def build_model(num_classes):
    """Construit MobileNetV3-Large avec une tête de classification à trois sorties."""
    base_model = tf.keras.applications.MobileNetV3Large(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(IMG_SHAPE, IMG_SHAPE, 3),
    )

    # MobileNetV3 de Keras contient sa propre couche de normalisation :
    # il attend des pixels bruts 0-255. Ne pas diviser par 255 et ne pas
    # appliquer les statistiques ImageNet (mean/std) avant le réseau.
    base_model.trainable = FINE_TUNING

    inputs = keras.Input(shape=(IMG_SHAPE, IMG_SHAPE, 3), name="coupe")
    x = base_model(inputs, training=FINE_TUNING)
    x = layers.Dense(1280, activation="relu", name="dense_1280")(x)
    x = layers.Dropout(0.2, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probabilites")(x)

    return keras.Model(inputs, outputs, name=MODEL_NAME), base_model

model, base_model = build_model(len(class_names))
model.summary()

### Compilation

L'optimiseur est **AdamW** avec un taux d'apprentissage de `1e-4` et un weight decay de `1e-4`. La perte est l'entropie croisée catégorielle parcimonieuse (`SparseCategoricalCrossentropy`), adaptée aux étiquettes entières et à une sortie softmax.

In [ ]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

trainable_params = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print(f"Paramètres : {model.count_params():,} | entraînables : {trainable_params:,}")
print(f"Optimiseur : AdamW | lr = 1e-4 | weight decay = 1e-4 | appareil = {DEVICE}")

## Entraînement

Deux callbacks sont utilisés :

- `ModelCheckpoint` conserve le meilleur modèle selon l'exactitude de validation ;
- `EarlyStopping` arrête l'entraînement après 5 époques sans amélioration et restaure les meilleurs poids.

Le modèle est enregistré dans `outputs/checkpoints/pulmoscan_mobilenetv3.keras`. En mode smoke test, le fichier porte le suffixe `_smoke` pour ne jamais écraser un vrai modèle.

In [ ]:
def format_duration(seconds):
    """Formate une durée en heures, minutes et secondes."""
    m, s = divmod(int(round(seconds)), 60)
    h, m = divmod(m, 60)
    return f"{h} h {m:02d} min {s:02d} s" if h else f"{m} min {s:02d} s"

checkpoint_path = CHECKPOINT_DIR / (
    f"{MODEL_NAME}_smoke.keras" if SMOKE_TEST else f"{MODEL_NAME}.keras"
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

t0 = time.perf_counter()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    shuffle=False,  # le jeu de données est déjà mélangé par tf.data
)
training_time = time.perf_counter() - t0

print(f"Entraînement terminé en {format_duration(training_time)}")
print("Meilleur modèle :", checkpoint_path)

## Résultats

Exactitude et perte sur les trois découpages. Les images de test n'ont jamais été utilisées pendant l'entraînement ni pour une décision sur le modèle.

Le jeu d'entraînement est évalué sans augmentation, avec le même recadrage central que la validation et le test. Cela donne une mesure plus lisible que l'évaluation sur les images augmentées.

> **Attention :** si `SMOKE_TEST = True`, ces chiffres proviennent d'un sous-ensemble réduit et n'ont aucune valeur scientifique. Ils servent seulement à vérifier que le pipeline fonctionne.

In [ ]:
def evaluate_split(name, ds):
    """Évalue le modèle sur un découpage et affiche perte et exactitude."""
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name:<13} perte = {loss:.4f} | exactitude = {acc:.4f} ({acc * 100:.2f} %)")
    return float(loss), float(acc)

train_loss, train_acc = evaluate_split("Entraînement", train_eval_ds)
val_loss,   val_acc   = evaluate_split("Validation", val_ds)
test_loss,  test_acc  = evaluate_split("Test", test_ds)

### Métriques par classe et matrice de confusion

L'exactitude globale peut masquer une classe mal reconnue. Cette cellule calcule, sur le jeu de test, la matrice de confusion et le rapport `classification_report` de scikit-learn : précision, rappel, F1 et support pour chaque classe.

Les figures sont enregistrées dans `assets/` avec des titres et des libellés en français :

- `assets/matrice_confusion.png` ;
- `assets/courbes_validation.png` (cellule suivante) ;
- `assets/predictions.png` (plus bas).

Les métriques sont aussi écrites dans `outputs/metrics.json`.

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

display_names = [class_to_name[c] for c in class_names]
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=display_names,
    output_dict=True,
    zero_division=0,
)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    cm,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar_kws={"label": "Nombre de coupes"},
    xticklabels=display_names,
    yticklabels=display_names,
    ax=ax,
)
ax.set_title("Matrice de confusion — jeu de test")
ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
fig.tight_layout()
fig.savefig(ASSETS_DIR / "matrice_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

rows = []
for name in display_names:
    r = report[name]
    rows.append({
        "Classe": name,
        "Précision": round(float(r["precision"]), 4),
        "Rappel": round(float(r["recall"]), 4),
        "F1": round(float(r["f1-score"]), 4),
        "Support": int(r["support"]),
    })

metrics_df = pd.DataFrame(rows)
display(metrics_df)

accuracy = float((y_true == y_pred).mean())
print(f"Exactitude globale : {accuracy:.4f} ({accuracy * 100:.2f} %)")

if SMOKE_TEST:
    print("Mode smoke test : ces métriques proviennent d'un sous-ensemble et ne doivent pas être publiées.")

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "accuracy": accuracy,
        "loss": test_loss,
        "confusion_matrix": cm.tolist(),
        "class_names": class_names,
        "per_class": rows,
        "smoke_test": SMOKE_TEST,
        "source": "PulmoScan AI — évaluation TensorFlow/Keras sur le jeu de test",
    }, f, ensure_ascii=False, indent=2)

print(f"Métriques enregistrées dans {OUTPUT_DIR / 'metrics.json'}")

### Courbes d'entraînement et de validation

Les courbes sont enregistrées dans `assets/courbes_validation.png` avec des libellés français.

In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

ax1.plot(epochs_range, history.history["loss"], marker="o", label="Entraînement")
ax1.plot(epochs_range, history.history["val_loss"], marker="o", label="Validation")
ax1.set_title("Perte par époque")
ax1.set_xlabel("Époque")
ax1.set_ylabel("Entropie croisée")
ax1.grid(color="#E3EAF1", lw=1)
ax1.legend()

ax2.plot(epochs_range, np.array(history.history["accuracy"]) * 100, marker="o", label="Entraînement")
ax2.plot(epochs_range, np.array(history.history["val_accuracy"]) * 100, marker="o", label="Validation")
ax2.set_title("Exactitude par époque")
ax2.set_xlabel("Époque")
ax2.set_ylabel("Exactitude (%)")
ax2.grid(color="#E3EAF1", lw=1)
ax2.legend()

for ax in (ax1, ax2):
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Courbes d'entraînement et de validation — TensorFlow/Keras", fontsize=15)
fig.tight_layout()
fig.savefig(ASSETS_DIR / "courbes_validation.png", dpi=150, bbox_inches="tight")
plt.show()

### Prédictions sur une coupe de test par classe

Cette figure montre une image de test par classe, la classe réelle et la prédiction du modèle avec sa probabilité. Elle est enregistrée dans `assets/predictions.png`.

In [ ]:
def process_image(image_path):
    """Prépare une image pour le modèle : 224 x 224, pixels bruts 0-255, forme [1, 224, 224, 3]."""
    raw = tf.io.read_file(str(image_path))
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image = tf.cast(image, tf.float32)  # pas de division par 255 : MobileNetV3 normalise en interne
    image = tf.image.resize(image, (256, 256), method="bilinear")
    image = tf.image.resize_with_crop_or_pad(image, IMG_SHAPE, IMG_SHAPE)
    return tf.expand_dims(image, axis=0)

fig, axes = plt.subplots(1, len(class_names), figsize=(5 * len(class_names), 5))
if len(class_names) == 1:
    axes = [axes]

for ax, cls_idx in zip(axes, range(len(class_names))):
    candidates = [p for p, y in zip(test_paths, test_labels) if y == cls_idx]
    if not candidates:
        ax.axis("off")
        ax.set_title(f"Aucune image pour {class_names[cls_idx]}")
        continue

    path = candidates[0]
    probs = model.predict(process_image(path), verbose=0)[0]
    pred_idx = int(probs.argmax())

    ax.imshow(Image.open(path).convert("RGB"))
    ax.axis("off")
    ax.set_title(
        f"Réel : {class_to_name[class_names[cls_idx]]}\n"
        f"Prédit : {class_to_name[class_names[pred_idx]]} ({probs[pred_idx] * 100:.1f} %)"
    )

fig.suptitle("Prédictions sur une coupe de test par classe", fontsize=15)
fig.tight_layout()
fig.savefig(ASSETS_DIR / "predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Sauvegarde et rechargement du modèle

Le modèle complet est enregistré au format `.keras`. Le fichier `class_names.json` contient l'ordre exact des classes. Pour recharger le modèle, il suffit de charger les deux fichiers et d'utiliser le même prétraitement.

In [ ]:
loaded_model = keras.models.load_model(checkpoint_path)
loaded_loss, loaded_acc = loaded_model.evaluate(test_ds, verbose=0)

print("Modèle rechargé :", checkpoint_path)
print(f"Test après rechargement : perte = {loaded_loss:.4f} | exactitude = {loaded_acc:.4f}")

## Prédire une image

`process_image` applique le même prétraitement que la validation : redimensionnement à 256 px, recadrage central à 224 px, pixels bruts 0-255. `predict` renvoie les probabilités softmax et les noms de classes dans l'ordre enregistré.

In [ ]:
def predict(image_path, model, topk=3):
    """Renvoie les k meilleures probabilités et les noms de classes associés."""
    probs = model.predict(process_image(image_path), verbose=0)[0]
    topk = min(topk, len(class_names))
    order = np.argsort(probs)[::-1][:topk]
    return probs[order], [class_names[i] for i in order]

idx = random.choice(range(len(test_paths)))
image_path = test_paths[idx]
y_true = test_labels[idx]

probs, names = predict(image_path, model, topk=len(class_names))

fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(7, 9),
    gridspec_kw={"height_ratios": [3, 2]},
)

ax1.imshow(Image.open(image_path).convert("RGB"))
ax1.set_title(f"Classe réelle : {class_to_name[class_names[y_true]]}")
ax1.axis("off")

y_pos = np.arange(len(probs))
ax2.barh(y_pos, probs, align="center")
ax2.set_yticks(y_pos)
ax2.set_yticklabels([class_to_name[n] for n in names])
ax2.set_xlim(0, 1)
ax2.set_xlabel("Probabilité")
ax2.set_title("Probabilité par classe")
ax2.invert_yaxis()

for i, p in enumerate(probs):
    ax2.text(min(p + 0.02, 0.98), i, f"{p * 100:.1f} %", va="center")

fig.tight_layout()
plt.show()

print("Image de test :", Path(image_path).relative_to(DATA_ROOT))
print("Classe réelle :", class_to_name[class_names[y_true]])
for p, n in zip(probs, names):
    print(f"{p:7.2%}  {class_to_name[n]}")

## Limites et analyse critique

- **Découpage par coupe, pas par patient.** Le jeu de données contient de nombreuses coupes par patient, et des coupes voisines se ressemblent beaucoup. Une même personne peut donc apparaître à la fois dans l'entraînement et dans le test, ce qui rend l'exactitude de test probablement trop optimiste.
- **Une seule source de données.** Toutes les coupes proviennent de deux hôpitaux de Wuhan. Des scanners, protocoles ou populations différentes peuvent produire des images très différentes, et le modèle n'a jamais été évalué dessus.
- **Coupes isolées.** Le modèle juge une coupe à la fois. Une évaluation clinique considère le scanner complet, les symptômes et les résultats biologiques.
- **Déséquilibre des classes.** La classe `nCT` représente environ la moitié des images. Une exactitude globale élevée peut donc cacher de moins bonnes performances sur `pCT` ou `NiCT` ; c'est pourquoi les métriques par classe sont importantes.
- **Absence de validation externe.** Aucun jeu de test indépendant, provenant d'un autre hôpital ou d'un autre appareil, n'est utilisé.
- **Résultats non reproduits dans ce dépôt.** Le notebook fournit le code d'évaluation, mais aucun entraînement complet sur le vrai jeu de données n'a été exécuté lors de la migration. Les résultats TensorFlow restent à produire.
- **Pas d'interprétabilité.** Le notebook ne montre pas encore de Grad-CAM ni de carte de saillance permettant de vérifier ce que le modèle regarde.

### Améliorations possibles

- Effectuer un découpage strict par patient.
- Évaluer le modèle sur un jeu de données externe.
- Examiner les prédictions avec Grad-CAM.
- Mesurer la calibration et fournir des intervalles de confiance.
- Publier les métriques détaillées par classe après un entraînement complet.

## Comparaison avec le projet d'origine

| Élément | PyTorch (projet d'origine) | TensorFlow/Keras (cette version) |
|---|---|---|
| Exactitude test | 99,76 % (résultat du projet d'origine) | à compléter après l'entraînement complet |
| F1 par classe | non fourni dans le projet d'origine | à compléter après l'entraînement complet |
| Durée d'entraînement | 41 min 07 s (25 époques, résultat du projet d'origine) | à mesurer |
| Matériel | MacBook Pro M5 Pro, 48 Go, MPS (projet d'origine) | à renseigner selon le run |

Les valeurs PyTorch sont reprises telles quelles du projet d'origine et clairement identifiées comme telles. Elles ne sont pas présentées comme des résultats de cette migration. Aucune métrique TensorFlow n'est inventée : la colonne de droite reste vide tant qu'un entraînement complet n'a pas été exécuté.

## Crédits et attribution

Ce projet est dérivé du dépôt open source d'origine : [URL_DU_REPO_ORIGINAL]. Le fichier `LICENSE` de ce dépôt est conservé.

- **Jeu de données :** Abu Zahid Bin Aziz, *CT Scans for COVID-19 Classification*, Kaggle, licence CC BY 4.0. Images originales issues de la ressource iCTCF : Ning, W. et al., *iCTCF: an integrative resource of chest computed tomography images and clinical features of patients with COVID-19 pneumonia* (2020).
- **Architecture :** Howard, A. et al., *Searching for MobileNetV3*, ICCV 2019.
- **Outils :** TensorFlow, Keras, NumPy, pandas, scikit-learn, Matplotlib, Seaborn, Pillow, Jupyter et Streamlit.

## Auteur

**Mohammed Oughlane**  
Marrakech, Maroc

- GitHub : [LIEN_GITHUB]
- LinkedIn : [LIEN_LINKEDIN]

## Avertissement

**PulmoScan AI est un projet pédagogique et ne constitue pas un dispositif médical.** Il ne doit pas être utilisé pour établir un diagnostic, trier des patients ou prendre une décision de santé. Les résultats présentés ne constituent pas une validation clinique et ne remplacent pas l'avis d'un professionnel de santé.